# CH12 project 3 — the camera through the filter

Project 1's camera and project 2's accelerator in one design. The camera writes
frames into DDR, software calls the filter on each one, and the result goes to
the DisplayPort.

**The filter is not in the video path.** That is worth being explicit about,
because it is the design decision the whole chapter turns on. A memory-mapped
accelerator sits beside the pipeline rather than inside it: it costs a DDR round
trip per frame and it cannot filter anything that is not already in memory — but
in exchange it does not care where the frame came from. The code below is
project 2's code with `vc.test_pattern(...)` replaced by `mipi.readframe()`, and
nothing else.

The camera is an **OV5647** — a Raspberry Pi Camera Module v1 — driven by
CH12's own `sw/ov5647.py` rather than by PYNQ's `Pcam5C` and its
`libpcam5c.so`, which only speak to the OV5640 in a Pcam 5C. Project 1 covers
why, and covers the 720p mode this sensor does not natively have and which
`ov5647.py` derives from its 2x2-binned readout at exactly 60 fps.

For this notebook none of that matters, which is the point: a memory-mapped
accelerator does not care where the pixels came from.

Pick the bitstream with `VARIANT`: `sv`, `vhdl` or `hls`.

Measured on hardware, filtering live camera frames in place:

| mode | per frame | rate |
|---|---|---|
| gray | 5.04 ms | 182.9 Mpixel/s |
| sobel | 5.04 ms | 182.9 Mpixel/s |
| invert | 5.04 ms | 182.9 Mpixel/s |
| colour | 5.03 ms | 183.2 Mpixel/s |

Within 0.1 ms of the 5.14 ms project 2 measured on a stored clip — the whole
argument for the memory-mapped interface in one number. Identical timing across
all four modes says it is DDR-bound: the Sobel costs exactly what a passthrough
costs.

One caveat worth knowing before you trust a run: on two occasions, on the first
call after a fresh PL download, the accelerator did not assert `AP_DONE` within
five seconds. Against an already-running design it took twenty-plus consecutive
calls without a miss. This is not understood. If `run_frames` raises
`TimeoutError` here, clear `CTRL` and retry rather than assuming the design is
broken.

In [ ]:
import pathlib, sys

# sw/ holds the software reference and the driver. In the repo it is two levels
# up; on the board everything is normally copied into one directory.
for _cand in ("../../sw", "../sw", "sw", "."):
    if (pathlib.Path(_cand) / "sobel_ref.py").exists():
        sys.path.insert(0, str(pathlib.Path(_cand).resolve()))
        break
else:
    raise FileNotFoundError("cannot find sobel_ref.py -- copy sw/*.py next to this notebook")

print("sw/ ->", sys.path[0])

In [ ]:
import pathlib
import sys
import time

import numpy as np
import PIL.Image
from IPython.display import display
from pynq import Overlay, allocate
from pynq.lib.video import VideoMode, DisplayPort, PIXEL_RGB

# CH12's camera driver. This import has to happen BEFORE the Overlay is
# constructed: PYNQ resolves hierarchy drivers newest-registered-first, and
# `pynq.lib.video` above has already registered `Pcam5C`, which matches the
# same hierarchy and would win otherwise.
for candidate in ("../../sw", "../sw", "."):
    if (pathlib.Path(candidate) / "ov5647.py").exists():
        sys.path.insert(0, candidate)
        break
import ov5647 as cam

import sobel_ref as ref
import video_clip as vc
from filter_driver import VideoFilter, frame_address

VARIANT = "sv"          # sv | vhdl | hls
W, H = 1280, 720

def find_bitstream(name, variant=None):
    """Locate a bitstream in the repo layout or beside this notebook.

    In the repo the builds land in ../out_<variant>/; on the board everything
    is normally copied into one directory. Try both rather than making the
    reader edit a path.
    """
    cands = []
    if variant:
        cands.append(f"../out_{variant}/{name}")
    cands += [f"../out/{name}", name]
    for c in cands:
        if pathlib.Path(c).exists():
            return c
    raise FileNotFoundError(f"{name} not found -- looked in {cands}")

BITSTREAM = find_bitstream("camera_sobel.bit", VARIANT)
print("overlay:", BITSTREAM)
try:
    ol = Overlay(BITSTREAM)
except OSError:
    # Removing a device-tree overlay leaks its `__symbols__` entries -- the
    # kernel warns about it at apply time -- so a later overlay declaring the
    # same axi_iic node is rejected with EINVAL, and PYNQ surfaces that as
    # "Device tree ... cannot be applied" or an OSError from the FPGA manager.
    # Every CH12 camera overlay declares that node, so going from project 0 or
    # 1 to this one needs a reboot. If the PL is ALREADY running this design,
    # attach to it instead of insisting on reprogramming.
    print("  reload refused (an overlay is already applied); attaching instead")
    ol = Overlay(BITSTREAM, download=False)
filt = VideoFilter(ol.video_filter_0)
mipi = ol.mipi
assert type(mipi).__name__ == "Ov5647Camera", (
    "Pcam5C won the registration race -- restart the kernel and make sure "
    "`import ov5647` runs before `Overlay(...)`")
# Before anything reads a video IP register. In this design they are at
# 0xA0000000, and held in reset they hang the board on the first access -- see
# project 1. configure() does this too; doing it here makes the notebook safe
# to re-run from any cell.
mipi.pipeline.release_video_reset()

sensor_mode = mipi.configure(VideoMode(W, H, 32))
mipi.start()
print(f"camera running: {sensor_mode.name} at {sensor_mode.fps:.1f} fps, "
      f"chip id {mipi.sensor.chip_id():#06x}")

# Raw Bayer is green until something balances it -- a Bayer sensor has twice as
# many green photosites, and nothing in the PL pipeline corrects for that. One
# grey-world pass; project 1 explains it. Skip this and the Sobel result is
# unaffected (it works on luma) but every colour-passthrough frame looks sick.
time.sleep(0.5)
r, g, b = mipi.auto_white_balance()
print(f"white balance : R {r:.2f}  G {g:.2f}  B {b:.2f}")

## One frame, four ways

The same captured frame through all four modes, side by side. Note that this is
the *same* accelerator, the *same* register map and the *same* driver call as
project 2 — only the source of the pixels changed.

In [ ]:
dst = allocate(shape=(H, W, 4), dtype=np.uint8)

mipi.readframe()                       # discard one; the first after start can be partial
shot = mipi.readframe()

for mode in ref.MODES:
    t = filt.run_frames(shot, dst, mode)
    dst.invalidate()
    print(f"{ref.MODE_NAMES[mode]:>8}  {t*1e3:6.2f} ms")
    display(PIL.Image.fromarray(np.array(dst)[:, :, [2, 1, 0]]).resize((400, 225)))

### Filtering a camera frame in place

`mipi.readframe()` returns a VDMA buffer with a physical address, so it can be
the accelerator's *source* directly — no copy on the way in either. The one
thing to watch is that the VDMA may hand the same buffer back around while the
accelerator is reading it; at four frame stores and single-digit-millisecond
filter times there is a wide margin, but it is a margin and not a guarantee.

In [ ]:
print("camera frame physical address:", hex(shot.device_address))
print("contiguous:", shot.flags["C_CONTIGUOUS"], " stride:", shot.strides[0], "expected", W*4)

## How close is the software reference on a real frame?

Not bit-exact, and it cannot be: the sensor is still exposing, so the frame the
software filters is not the frame the hardware filtered. That is exactly why
project 2 uses a generated clip for the exact comparison. Here the useful check
is that the two agree to within sensor noise.

In [ ]:
captured = np.array(shot)              # snapshot it, so both filter the SAME pixels
filt.run_frames(shot, dst, ref.MODE_SOBEL)
dst.invalidate()

expected = ref.filter_frame(captured, ref.MODE_SOBEL)
diff = np.abs(np.array(dst)[:, :, 1].astype(np.int16) - expected[:, :, 1].astype(np.int16))
print(f"differing samples : {int(np.count_nonzero(diff))}")
print(f"max difference    : {int(diff.max())}")
print()
print("Zero here means the VDMA did not touch the buffer while the accelerator")
print("was reading it. Non-zero does not mean the filter is wrong -- rerun the")
print("cell; the exact check lives in project 2 where the input holds still.")

## Live, on the screen

The camera into the filter into the DisplayPort, with no copies in either
direction: the accelerator reads the VDMA's buffer and writes the display's.

Change `MODE` and rerun — the filter is reconfigured per frame anyway, so
switching mode costs one register write.

In [ ]:
# The DisplayPort on this board offers only 24bpp modes. Asking for 32 --
# VideoMode(W, H, 32) -- is refused outright with "is not supported", and
# handing configure() an EDID mode together with PIXEL_RGB yields a
# 3-channel view over a 4-byte stride, which is not something the accelerator
# can write in place. PIXEL_RGB is DRM_FORMAT_RGB888, whose memory order is
# B,G,R -- the same order the filter and the camera use.
dp = DisplayPort()
wanted = [m for m in dp.modes if m.width == W and m.height == H]
if not wanted:
    dp.close()
    raise RuntimeError(f"monitor does not offer {W}x{H}; "
                       f"it offers {sorted({(m.width, m.height) for m in dp.modes})}")
dp_mode = max(wanted, key=lambda m: m.fps)
dp.configure(dp_mode, PIXEL_RGB)
print(f"DisplayPort {dp_mode.width}x{dp_mode.height} @ {dp_mode.fps} Hz, "
      f"{dp_mode.bits_per_pixel} bpp")

In [ ]:
import cv2
MODE = ref.MODE_SOBEL
N = 200

frame = dp.newframe()
t0 = time.perf_counter()
for _ in range(N):
    filt.run_frames(mipi.readframe(), dst, MODE)
    dst.invalidate()
    cv2.cvtColor(np.asarray(dst), cv2.COLOR_BGRA2BGR, dst=frame)
    dp.writeframe(frame)
    frame = dp.newframe()
pl_fps = N / (time.perf_counter() - t0)
print(f"camera -> PL -> screen : {pl_fps:.1f} fps")

## The same thing in software

Identical loop, identical source, identical display. The only change is where
the filtering happens — and this is the comparison the chapter exists to make.

In [ ]:
N = 60          # fewer, because this one is slow

frame = dp.newframe()
t0 = time.perf_counter()
for _ in range(N):
    out = ref.filter_frame_opencv(np.array(mipi.readframe()), MODE)
    cv2.cvtColor(out, cv2.COLOR_BGRA2BGR, dst=frame)
    dp.writeframe(frame)
    frame = dp.newframe()
cv_fps = N / (time.perf_counter() - t0)
print(f"camera -> OpenCV -> screen : {cv_fps:.1f} fps")

frame = dp.newframe()
t0 = time.perf_counter()
for _ in range(20):
    out = ref.filter_frame(np.array(mipi.readframe()), MODE)
    cv2.cvtColor(out, cv2.COLOR_BGRA2BGR, dst=frame)
    dp.writeframe(frame)
    frame = dp.newframe()
np_fps = 20 / (time.perf_counter() - t0)
print(f"camera -> NumPy  -> screen : {np_fps:.1f} fps")

print()
print(f"{'implementation':<16}{'fps':>8}{'vs camera 60fps':>18}")
for name, fps in (("PL", pl_fps), ("OpenCV", cv_fps), ("NumPy (exact)", np_fps)):
    verdict = "keeps up" if fps >= 59 else f"drops {60/fps:.1f}x"
    print(f"{name:<16}{fps:8.1f}{verdict:>18}")

## Mode switching, live

A mode change is one 32-bit register write, latched when the next frame starts.
This cycles through all four while the screen is running, which is the most
direct demonstration that the accelerator is being reconfigured per frame rather
than rebuilt.

In [ ]:
# The filter needs four bytes per pixel, and a 24bpp DisplayPort frame has
# three -- so it cannot be the accelerator's destination. Filter into `dst` and
# convert on the way out, exactly as the timed loop above does.
frame = dp.newframe()
for mode in ref.MODES:
    print(f"{ref.MODE_NAMES[mode]} ...")
    t_end = time.time() + 3.0
    while time.time() < t_end:
        filt.run_frames(mipi.readframe(), dst, mode)
        dst.invalidate()
        cv2.cvtColor(np.asarray(dst), cv2.COLOR_BGRA2BGR, dst=frame)
        dp.writeframe(frame)
        frame = dp.newframe()
print("done")

## Clean up

In [ ]:
dp.close()
mipi.close()          # stops the VDMA, parks the MIPI lanes, releases the I2C bus
dst.freebuffer()
print("released")